In [1]:
# ============================================================
# 0. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from scipy.stats import rankdata

SEED = 42
N_SPLITS = 5

In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (594194, 21)
Test shape : (254655, 20)


In [3]:
train["Churn"] = train["Churn"].map({"Yes": 1, "No": 0})

y = train["Churn"]
train.drop(["Churn"], axis=1, inplace=True)

In [4]:
all_data = pd.concat([train, test], axis=0, ignore_index=True)

# -------------------------
# Financial Features
# -------------------------
all_data["AvgChargePerMonth"] = (
    all_data["TotalCharges"] / (all_data["tenure"] + 1)
)

all_data["ChargeDiff"] = (
    all_data["MonthlyCharges"] - all_data["AvgChargePerMonth"]
)

# -------------------------
# Contract Ordinal Encoding
# -------------------------
contract_map = {
    "Month-to-month": 0,
    "One year": 1,
    "Two year": 2
}
all_data["ContractOrdinal"] = all_data["Contract"].map(contract_map)

# -------------------------
# Tenure Bucketing
# -------------------------
all_data["TenureGroup"] = pd.cut(
    all_data["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=False
)

# -------------------------
# Number of Services
# -------------------------
services = [
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

all_data["NumServices"] = (
    (all_data[services] != "No").sum(axis=1)
)

In [5]:
categorical_cols = all_data.select_dtypes(include="object").columns

for col in categorical_cols:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col])

In [6]:
train = all_data.iloc[:len(y)].copy()
test  = all_data.iloc[len(y):].copy()

X = train.drop(["id"], axis=1)
test_ids = test["id"]
test = test.drop(["id"], axis=1)

In [7]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

In [8]:
lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 64,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_samples': 50,
    'reg_alpha': 0.5,
    'reg_lambda': 1.5,
    'objective': 'binary',
    'metric': 'auc',
    'random_state': SEED
}

lgb_oof = np.zeros(len(X))
lgb_test = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"\nLGB Fold {fold+1}")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(200)]
    )
    
    lgb_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    lgb_test += model.predict_proba(test)[:, 1] / N_SPLITS

print("LGB CV:", roc_auc_score(y, lgb_oof))


LGB Fold 1
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014310 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1152
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[706]	valid_0's auc: 0.915906

LGB Fold 2
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009567 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_w

In [37]:
cat_params = {
    'iterations': 2000,
    'learning_rate': 0.03,
    'depth': 8,
    'l2_leaf_reg': 3,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'verbose': 200,
    'task_type': 'GPU'
}

cat_oof = np.zeros(len(X))
cat_test = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    
    print(f"\nCAT Fold {fold+1}")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model = CatBoostClassifier(**cat_params)
    
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=200
    )
    
    cat_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    cat_test += model.predict_proba(test)[:, 1] / N_SPLITS

print("CAT CV:", roc_auc_score(y, cat_oof))


CAT Fold 1


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9002952	best: 0.9002952 (0)	total: 218ms	remaining: 7m 16s
200:	test: 0.9139778	best: 0.9139778 (200)	total: 2.98s	remaining: 26.7s
400:	test: 0.9149423	best: 0.9149423 (400)	total: 5.63s	remaining: 22.4s
600:	test: 0.9153189	best: 0.9153189 (600)	total: 8.26s	remaining: 19.2s
800:	test: 0.9155481	best: 0.9155481 (800)	total: 10.9s	remaining: 16.3s
1000:	test: 0.9156366	best: 0.9156382 (999)	total: 13.5s	remaining: 13.5s
1200:	test: 0.9156936	best: 0.9156948 (1183)	total: 16.2s	remaining: 10.8s
1400:	test: 0.9157231	best: 0.9157278 (1389)	total: 18.9s	remaining: 8.06s
1600:	test: 0.9157121	best: 0.9157315 (1440)	total: 21.5s	remaining: 5.36s
bestTest = 0.9157314599
bestIteration = 1440
Shrink model to first 1441 iterations.

CAT Fold 2


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9016700	best: 0.9016700 (0)	total: 14ms	remaining: 28.1s
200:	test: 0.9152065	best: 0.9152065 (200)	total: 2.71s	remaining: 24.3s
400:	test: 0.9160785	best: 0.9160785 (400)	total: 5.34s	remaining: 21.3s
600:	test: 0.9164158	best: 0.9164182 (599)	total: 7.97s	remaining: 18.6s
800:	test: 0.9166161	best: 0.9166161 (800)	total: 10.6s	remaining: 15.9s
1000:	test: 0.9167064	best: 0.9167064 (1000)	total: 13.3s	remaining: 13.3s
1200:	test: 0.9167593	best: 0.9167624 (1188)	total: 15.9s	remaining: 10.6s
1400:	test: 0.9167712	best: 0.9167768 (1288)	total: 18.6s	remaining: 7.95s
bestTest = 0.9167768061
bestIteration = 1288
Shrink model to first 1289 iterations.

CAT Fold 3


...

0:	test: 0.9019100	best: 0.9019100 (0)	total: 14.7ms	remaining: 29.3s
200:	test: 0.9146552	best: 0.9146552 (200)	total: 2.69s	remaining: 24.1s
400:	test: 0.9154390	best: 0.9154390 (400)	total: 5.34s	remaining: 21.3s
600:	test: 0.9157934	best: 0.9157934 (600)	total: 8.05s	remaining: 18.7s
800:	test: 0.9159718	best: 0.9159718 (800)	total: 10.7s	remaining: 16s
1000:	test: 0.9160812	best: 0.9160812 (1000)	total: 13.3s	remaining: 13.3s
1200:	test: 0.9161195	best: 0.9161195 (1200)	total: 15.9s	remaining: 10.6s
1400:	test: 0.9161214	best: 0.9161369 (1230)	total: 18.6s	remaining: 7.95s
bestTest = 0.9161368608
bestIteration = 1230
Shrink model to first 1231 iterations.

CAT Fold 4


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9023281	best: 0.9023281 (0)	total: 14ms	remaining: 28s
200:	test: 0.9155696	best: 0.9155696 (200)	total: 2.73s	remaining: 24.4s
400:	test: 0.9164550	best: 0.9164550 (400)	total: 5.38s	remaining: 21.4s
600:	test: 0.9168709	best: 0.9168709 (600)	total: 8.02s	remaining: 18.7s
800:	test: 0.9170547	best: 0.9170547 (800)	total: 10.7s	remaining: 16s
1000:	test: 0.9171590	best: 0.9171590 (1000)	total: 13.3s	remaining: 13.3s
1200:	test: 0.9172370	best: 0.9172373 (1199)	total: 15.9s	remaining: 10.6s
1400:	test: 0.9172223	best: 0.9172375 (1201)	total: 18.6s	remaining: 7.96s
bestTest = 0.9172374606
bestIteration = 1201
Shrink model to first 1202 iterations.

CAT Fold 5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9003445	best: 0.9003445 (0)	total: 14.4ms	remaining: 28.9s
200:	test: 0.9129160	best: 0.9129160 (200)	total: 2.72s	remaining: 24.3s
400:	test: 0.9138928	best: 0.9138928 (400)	total: 5.36s	remaining: 21.4s
600:	test: 0.9142872	best: 0.9142879 (599)	total: 8.01s	remaining: 18.6s
800:	test: 0.9144704	best: 0.9144704 (800)	total: 10.6s	remaining: 15.9s
1000:	test: 0.9145456	best: 0.9145456 (1000)	total: 13.3s	remaining: 13.2s
1200:	test: 0.9145917	best: 0.9145924 (1188)	total: 15.9s	remaining: 10.6s
1400:	test: 0.9146182	best: 0.9146259 (1332)	total: 18.6s	remaining: 7.93s
bestTest = 0.9146258533
bestIteration = 1332
Shrink model to first 1333 iterations.
CAT CV: 0.9160915735905899


In [38]:
final_test = (
   
    0.3 * lgb_test +
    0.3 * cat_test
)

In [39]:
final_test = (
    
    rankdata(lgb_test) +
    rankdata(cat_test)
) / 3

In [40]:
ensemble_oof = (
   
    0.3 * lgb_oof +
    0.3 * cat_oof
)

print("\nFINAL ENSEMBLE CV:", roc_auc_score(y, ensemble_oof))


FINAL ENSEMBLE CV: 0.916463024732951


In [41]:
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": final_test
})

submission.to_csv("submission_01.csv", index=False)
print("Submission saved.")

Submission saved.
